# Laya on messages that fit none of your options

What does a choice question do when the message does not belong to any option? A thank-you note sent to a support inbox is a good test: the options are *bug, how-to, feature request, account access*, and the message is none of them.

This notebook runs the original **Laya English** model (`convaiinnovations/laya`, Apache-2.0) in PyTorch on five short tickets, first without and then with an explicit "thanks" option. It goes with the [layaForWeb](https://github.com/vishalmysore/layaForWeb) demo.

**No GPU needed.** The default CPU runtime is enough. The first run downloads the 843 MB checkpoint from Hugging Face; no token is required.

In [ ]:
!pip -q install laya

In [ ]:
import torch, laya

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
agent = laya.load("convaiinnovations/laya", device=device)

## The five tickets

A and B are pure thank-you notes. C is a thank-you that also asks a real question. D is praise plus a feature request. E is a real emergency, for contrast.

In [ ]:
def ticket(subject, text):
    return {"ticket": {"subject": subject, "text": text}}

states = {
    "A thank-you (mentions a demo)": ticket("Working well after the update",
        "Since the last update, the app is working really well. I have a demo in one hour and I want to thank you!"),
    "B praise only": ticket("Well done",
        "Great product, love the new dashboard. Nothing to report, just wanted to say well done to the team."),
    "C thanks + real question": ticket("Thanks, and a question",
        "Thanks for the quick fix yesterday, everything works now! Just wondering if there is a way to schedule reports to run automatically."),
    "D praise + feature request": ticket("Love it",
        "Your app is fantastic. Could you please add dark mode?"),
    "E real emergency": ticket("Cannot log in",
        "I can't log in and I have a demo in one hour. Please help!"),
}

## 1. Without a "thanks" option

A choice question has to spread its probability across the options it is given. With no option that fits, the model still picks one, so look at the confidence, not only the top answer.

In [ ]:
team_without = {"type": "choice", "instructions": "Which team should handle this message?",
                "criteria": {"bug": "Something is broken",
                             "how_to": "A usage question",
                             "feature_request": "A request for a new capability",
                             "account_access": "Login, password or account access"}}

def show_team(question, states):
    print(f"{'ticket':32} {'top option':16} {'prob':>5} {'confidence':>10}   second")
    for name, s in states.items():
        a = agent.system_one(s, {"team": question})["answers"]["team"]
        ranked = sorted(a["probabilities"].items(), key=lambda kv: -kv[1])
        (t1, p1), (t2, p2) = ranked[0], ranked[1]
        print(f"{name:32} {t1:16} {p1:5.2f} {a['confidence']:10.2f}   {t2} {p2:.2f}")

show_team(team_without, states)

Both pure thank-you notes (A and B) get forced into `bug`, but with very low confidence (about 0.15 and 0.06), and that low confidence is the useful signal: a person should look at them. Ticket C is forced into `bug` too, at 0.74 with a confidence of 0.41.

## 2. With an explicit "thanks" option

In [ ]:
team_with = {"type": "choice", "instructions": "Which team should handle this message?",
             "criteria": {**team_without["criteria"],
                          "thanks": "Praise or a thank-you, nothing to fix"}}

show_team(team_with, states)

On the PyTorch model this gave `thanks` at about 96% for A and 91% for B, and `feature_request` for D and `account_access` for E.

**C is a real failure:** a message that thanks *and* asks a question is still called a thank-you at about 94%. If your data has mixed messages, ask separate questions (for example "does this contain a question?") and check the mixed cases by hand.

## 3. The urgency question

In [ ]:
urgency = {"type": "score", "instructions": "How urgent is it for our team to act?",
           "criteria": ["No action needed", "Can wait", "Today", "Right now"]}

print(f"{'ticket':32} {'score':>5}  most likely level    confidence")
for name, s in states.items():
    a = agent.system_one(s, {"urgency": urgency})["answers"]["urgency"]
    lvl = max(a["probabilities"], key=a["probabilities"].get)
    print(f"{name:32} {a['score']:5.2f}  {a['legend'][lvl]:18} {a['confidence']:10.2f}")

Ticket A says "I have a demo in one hour" inside a thank-you. The model reads the phrase and not the mood, so urgency leans high. It also rates the dark-mode request (D) as "Right now". All five urgency confidences are low (0.10 to 0.36), so at a threshold of 0.90 none of them would be acted on automatically. Read the probabilities and not only the top level.

## 4. Yes/no questions on this text

Different wordings of "does this need action?" give different answers. Try your own wording here.

In [ ]:
wordings = {
    "asks_for_help":       "The customer is asking for help or for something to be done",
    "wants_reply":         "The customer is waiting for an answer from us",
    "just_thanks":         "The message is only a thank-you and asks for nothing",
    "problem_or_question": "The message contains a problem to fix or a question to answer",
}
qs = {k: {"type": "noul", "instructions": v} for k, v in wordings.items()}

print(f"{'P(yes)':32}" + "".join(f"{k[:16]:>18}" for k in wordings))
for name, s in states.items():
    r = agent.system_one(s, qs)["answers"]
    print(f"{name:32}" + "".join(f"{r[k]['noul']:18.2f}" for k in wordings))

On the PyTorch model none of these wordings was right on all five tickets. `problem_or_question` was the best, but it still missed C. Yes/no questions were the least reliable question type on text like this, so prefer a choice question with an explicit "nothing to do" option, and test any yes/no wording on your own examples before using it.

## Your own message

In [ ]:
my_text = "Thank you for the great support last week!"   # <- edit this

a = agent.system_one(ticket("My message", my_text), {"team": team_with})["answers"]["team"]
for label, p in sorted(a["probabilities"].items(), key=lambda kv: -kv[1]):
    print(f"{label:16} {p:.2f}")
print("confidence:", round(a["confidence"], 2))